# المرحلة B: تدريب 3 موديلات لتنبؤ `plant_pu` من طقس نفس اللحظة
GBM وMLP وAE، والتدريب والـ early stopping على صفوف `clean==1` بس (train للتدريب، val للـ early stopping).
الـ test بيتقرا للتقرير بس ومفيش أي ضبط عليه. المرحلة دي مفيهاش thresholds ولا scores ولا metrics للشذوذ.

In [1]:
import copy, gzip, hashlib, json, os, platform, random, sys, time
import numpy as np
import pandas as pd
import warnings
import lightgbm as lgb
import torch
# LightGBM 4.7 بيطلّع تحذير deprecation على اسم وسيط eval_set (مفيش تأثير على النتايج)
warnings.filterwarnings("ignore", message=".*eval_set.*deprecated.*")
from torch import nn

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
VERSIONS = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
            "torch": torch.__version__, "lightgbm": lgb.__version__, "platform": platform.platform()}
print(VERSIONS)

A_DIR, OUT_DIR = "phase_a", "."
for d in ("preds", "models"):
    os.makedirs(f"{OUT_DIR}/{d}", exist_ok=True)
SPLITS = ("train", "val", "test")
FEATURES = ["irr_wm2", "temp_c", "mod_temp_c", "solar_elev_deg", "hour_sin", "hour_cos", "doy_sin", "doy_cos"]
TARGET = "plant_pu"
MASKS = ["scoreable", "clean", "in_event_margin", "split"]
EXPECTED_CLEAN = {"train": 83135, "val": 12146, "test": 8857}
OFFSET_START = pd.Timestamp("2022-08-21", tz="UTC")
NET_SEEDS = (42, 43, 44)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

{'python': '3.12.3', 'pandas': '3.0.2', 'numpy': '2.4.4', 'torch': '2.14.1+cu130', 'lightgbm': '4.7.0', 'platform': 'Linux-6.18.44-fc-v153-x86_64-with-glibc2.39'}


## B0. التحميل والتحقق

In [2]:
tab = {s: pd.read_csv(f"{A_DIR}/table_{s}.csv.gz", index_col="time_utc", parse_dates=True) for s in SPLITS}
scaler = json.load(open(f"{A_DIR}/scaler_clean.json"))
dsum = json.load(open(f"{A_DIR}/dataset_summary.json"))
for s in SPLITS:
    print(s, tab[s].shape, list(tab[s].columns))
    missing = [c for c in FEATURES + [TARGET] + MASKS if c not in tab[s].columns]
    assert not missing, f"missing columns in {s}: {missing}"
print("scaler columns:", scaler["columns"])

P = scaler["params"]
def scale_x(df):
    return ((df[FEATURES] - pd.Series({c: P[c]["median"] for c in FEATURES})) / pd.Series({c: P[c]["scale"] for c in FEATURES})).to_numpy(np.float32)
def scale_y(df):
    return ((df[TARGET] - P[TARGET]["median"]) / P[TARGET]["scale"]).to_numpy(np.float32)
def unscale_y(v):
    return v * P[TARGET]["scale"] + P[TARGET]["median"]

# عدد صفوف clean وعدم وجود NaN، وعدم وجود NaN في صفوف scoreable
for s in SPLITS:
    t = tab[s]
    c = t[t["clean"] == 1]
    assert len(c) == EXPECTED_CLEAN[s], f"clean rows in {s}: {len(c)}"
    assert c[FEATURES + [TARGET]].notna().all().all(), f"NaN in clean rows of {s}"
    sc = t[t["scoreable"] == 1]
    assert sc[FEATURES + [TARGET]].notna().all().all(), f"NaN in scoreable rows of {s}"
    assert ((t["clean"] == 1) <= (t["scoreable"] == 1)).all()
assert {r["split"]: r["clean rows"] for r in dsum["clean_table"]} == EXPECTED_CLEAN
print("B0 asserts passed | clean rows:", {s: int(tab[s]["clean"].sum()) for s in SPLITS},
      "| scoreable rows:", {s: int(tab[s]["scoreable"].sum()) for s in SPLITS})

# المصفوفات المقيّسة
clean = {s: tab[s][tab[s]["clean"] == 1] for s in SPLITS}
Xtr, ytr = scale_x(clean["train"]), scale_y(clean["train"])
Xva, yva = scale_x(clean["val"]), scale_y(clean["val"])
assert np.isfinite(Xtr).all() and np.isfinite(Xva).all()

train (210439, 13) ['irr_wm2', 'temp_c', 'mod_temp_c', 'solar_elev_deg', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'plant_pu', 'scoreable', 'clean', 'in_event_margin', 'split']
val (35040, 13) ['irr_wm2', 'temp_c', 'mod_temp_c', 'solar_elev_deg', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'plant_pu', 'scoreable', 'clean', 'in_event_margin', 'split']
test (35033, 13) ['irr_wm2', 'temp_c', 'mod_temp_c', 'solar_elev_deg', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'plant_pu', 'scoreable', 'clean', 'in_event_margin', 'split']
scaler columns: ['irr_wm2', 'temp_c', 'mod_temp_c', 'solar_elev_deg', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'plant_pu']
B0 asserts passed | clean rows: {'train': 83135, 'val': 12146, 'test': 8857} | scoreable rows: {'train': 86985, 'val': 12214, 'test': 12180}


## B1. فحص الإزاحة (للتسجيل فقط)
مقارنة توزيع `irr_wm2` (W/m² الخام) في clean train وclean val قبل 2022-08-21 وclean val جوا فترة الإزاحة.

In [3]:
cv = clean["val"]
groups = {"clean train": clean["train"], "clean val before offset": cv[cv.index < OFFSET_START],
          "clean val inside offset": cv[cv.index >= OFFSET_START]}
assert len(groups["clean val inside offset"]) == 4659
rows = []
for k, g in groups.items():
    q = g["irr_wm2"].quantile([0.25, 0.5, 0.75])
    rows.append({"set": k, "rows": len(g), "irr Q1": round(q[0.25], 2), "irr median": round(q[0.5], 2), "irr Q3": round(q[0.75], 2),
                 "irr mean": round(g["irr_wm2"].mean(), 2)})
b1 = pd.DataFrame(rows); print(b1.to_string(index=False))

                    set  rows  irr Q1  irr median  irr Q3  irr mean
            clean train 83135  238.79      508.51  785.76    515.45
clean val before offset  7487  280.69      612.51  844.63    568.41
clean val inside offset  4659  302.70      553.32  778.46    538.94


## B2. GBM (LightGBM)
L2، الـ target المقيّس، الـ 8 features. `learning_rate=0.05` و`min_data_in_leaf=100` و`feature_fraction=1.0` و`n_estimators=2000` وearly stopping 50 جولة على clean val.
`num_leaves` ∈ {15, 31, 63} والاختيار بأقل MSE على clean val.

In [4]:
gbm_runs, gbm_models = {}, {}
for nl in (15, 31, 63):
    m = lgb.LGBMRegressor(objective="regression", learning_rate=0.05, num_leaves=nl, min_child_samples=100,
                          colsample_bytree=1.0, n_estimators=2000, random_state=SEED, deterministic=True,
                          force_row_wise=True, n_jobs=1, verbose=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="l2",
          callbacks=[lgb.early_stopping(50, verbose=False)])
    gbm_runs[nl] = {"best_iteration": int(m.best_iteration_), "val_mse_scaled": float(m.best_score_["valid_0"]["l2"])}
    gbm_models[nl] = m
print(pd.DataFrame(gbm_runs).T.to_string())
best_nl = min(gbm_runs, key=lambda k: gbm_runs[k]["val_mse_scaled"])
gbm = gbm_models[best_nl]
print("selected num_leaves:", best_nl)
gain = gbm.booster_.feature_importance(importance_type="gain")
imp = pd.Series(100 * gain / gain.sum(), index=FEATURES).round(2)
print("feature importance (gain %):"); print(imp.to_string())

    best_iteration  val_mse_scaled
15           120.0        0.004094
31            92.0        0.004073
63            99.0        0.004330
selected num_leaves: 31
feature importance (gain %):
irr_wm2           98.72
temp_c             0.39
mod_temp_c         0.19
solar_elev_deg     0.12
hour_sin           0.15
hour_cos           0.06
doy_sin            0.29
doy_cos            0.07


## B3. AE وMLP (نفس الكلاس `Net`)
encoder: `8 → 32 → 16 → 4` وdropout 0.15 بعد الأولى، decoder: `4 → 16 → 32 → 8`، head: `4 → 8 → 1`.
`loss = λ_recon × MSE(x̂, x) + λ_pred × MSE(ŷ, y)`: الـ AE بـ (1، 5) والـ MLP بـ (0، 5).
Adam بـ `lr=1e-3` وbatch 512 وحد أقصى 50 epoch، والـ early stopping على أقل pred MSE على clean val بـ patience 5.

In [5]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Dropout(0.15), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 4))
        self.dec = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 8))
        self.head = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), self.head(z).squeeze(-1)

LAMBDAS = {"ae": (1.0, 5.0), "mlp": (0.0, 5.0)}
Xtr_t, ytr_t, Xva_t, yva_t = map(torch.from_numpy, (Xtr, ytr, Xva, yva))
mse = nn.functional.mse_loss

@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    xr, yh = model(X)
    return float(mse(yh, y)), float(mse(xr, X))

def train_run(kind, seed, max_epochs=50, patience=5):
    lam_r, lam_p = LAMBDAS[kind]
    torch.manual_seed(seed)
    model = Net()
    init_state = copy.deepcopy(model.state_dict())
    torch.manual_seed(seed)                         # نفس الـ RNG stream (dropout) للاتنين
    gen = torch.Generator().manual_seed(seed)       # نفس ترتيب الـ batches للاتنين
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    n = len(Xtr_t)
    curves = {"train_total_loss": [], "train_pred_mse": [], "val_pred_mse": [], "val_recon_mse": []}
    best, best_state, best_epoch, bad = np.inf, None, 0, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=gen)
        tot = 0.0
        for i in range(0, n, 512):
            b = perm[i:i + 512]
            xr, yh = model(Xtr_t[b])
            loss = lam_r * mse(xr, Xtr_t[b]) + lam_p * mse(yh, ytr_t[b])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        tr_p, _ = evaluate(model, Xtr_t, ytr_t)
        va_p, va_r = evaluate(model, Xva_t, yva_t)
        curves["train_total_loss"].append(tot / n); curves["train_pred_mse"].append(tr_p)
        curves["val_pred_mse"].append(va_p); curves["val_recon_mse"].append(va_r)
        assert np.isfinite([tot, tr_p, va_p, va_r]).all(), f"NaN/inf in {kind} seed {seed} epoch {ep}"
        if va_p < best:
            best, best_state, best_epoch, bad = va_p, copy.deepcopy(model.state_dict()), ep, 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, init_state, {"kind": kind, "seed": seed, "lambda_recon": lam_r, "lambda_pred": lam_p,
                               "best_epoch": best_epoch, "epochs_run": len(curves["val_pred_mse"]),
                               "best_val_pred_mse_scaled": best, "curves": curves}

# smoke test: 3 epochs على الـ AE
_, _, smoke = train_run("ae", SEED, max_epochs=3, patience=99)
sm = smoke["curves"]
assert len(sm["train_total_loss"]) == 3 and sm["train_total_loss"][2] < sm["train_total_loss"][0]
assert np.isfinite(sm["val_pred_mse"]).all()
print("smoke test (AE, seed 42, 3 epochs): train_total_loss", [round(v, 5) for v in sm["train_total_loss"]],
      "| val_pred_mse", [round(v, 5) for v in sm["val_pred_mse"]])
SMOKE = {k: v for k, v in smoke.items() if k != "curves"} | {"curves": sm}

smoke test (AE, seed 42, 3 epochs): train_total_loss [0.89896, 0.22484, 0.13266] | val_pred_mse [0.01072, 0.00698, 0.00569]


In [6]:
nets, runs_log = {}, []
t0 = time.time()
for seed in NET_SEEDS:
    for kind in ("ae", "mlp"):
        model, init_state, log = train_run(kind, seed)
        nets[(kind, seed)] = model
        runs_log.append(log)
        log["_init"] = init_state
    # نفس الـ init للاتنين، والـ decoder في الـ MLP ما اتغيّرش
    a, m = runs_log[-2]["_init"], runs_log[-1]["_init"]
    assert all(torch.equal(a[k], m[k]) for k in a), f"AE and MLP init differ for seed {seed}"
    dec_keys = [k for k in m if k.startswith("dec.")]
    assert all(torch.equal(m[k], nets[("mlp", seed)].state_dict()[k]) for k in dec_keys), "MLP decoder was updated"
for r in runs_log:
    r.pop("_init")
print(f"training time: {time.time() - t0:.0f}s")
summ = pd.DataFrame([{"kind": r["kind"], "seed": r["seed"], "best_epoch": r["best_epoch"], "epochs_run": r["epochs_run"],
                      "best val pred MSE (scaled)": round(r["best_val_pred_mse_scaled"], 5),
                      "train pred MSE at best": round(r["curves"]["train_pred_mse"][r["best_epoch"] - 1], 5),
                      "val recon MSE at best": round(r["curves"]["val_recon_mse"][r["best_epoch"] - 1], 5)} for r in runs_log])
print(summ.to_string(index=False))
print("B3 asserts passed: identical init for AE/MLP per seed; MLP decoder unchanged")

training time: 42s
kind  seed  best_epoch  epochs_run  best val pred MSE (scaled)  train pred MSE at best  val recon MSE at best
  ae    42           8          13                     0.00396                 0.00479                0.02420
 mlp    42          14          19                     0.00369                 0.00441                0.39523
  ae    43           7          12                     0.00446                 0.00515                0.03388
 mlp    43           4           9                     0.00424                 0.00496                0.40181
  ae    44          23          28                     0.00355                 0.00460                0.02050
 mlp    44           9          14                     0.00446                 0.00466                0.41466
B3 asserts passed: identical init for AE/MLP per seed; MLP decoder unchanged


## B4. التنبؤات
ŷ بالـ p.u. (`ŷ × scale + median`) على كل صفوف `scoreable==1` في val وtest. الملفات: `preds/{model}_seed{k}_{split}.csv.gz`.

In [7]:
@torch.no_grad()
def net_predict(model, X):
    model.eval()
    return model(torch.from_numpy(X))[1].numpy()

def predict_scaled(name, seed, X):
    return gbm.predict(X) if name == "gbm" else net_predict(nets[(name, seed)], X)

MODELS = [("gbm", SEED)] + [(k, s) for k in ("mlp", "ae") for s in NET_SEEDS]
pred_files = []
for name, seed in MODELS:
    for s in ("val", "test"):
        sc = tab[s][tab[s]["scoreable"] == 1]
        yhat = unscale_y(predict_scaled(name, seed, scale_x(sc)).astype(np.float64))
        out = pd.DataFrame({"y_pu": sc[TARGET].to_numpy(), "yhat_pu": yhat, "clean": sc["clean"].to_numpy(),
                            "in_event_margin": sc["in_event_margin"].to_numpy()}, index=sc.index)
        assert len(out) == int(tab[s]["scoreable"].sum()) and np.isfinite(out[["y_pu", "yhat_pu"]].to_numpy()).all()
        assert int(out["clean"].sum()) == EXPECTED_CLEAN[s]
        fn = f"preds/{name}_seed{seed}_{s}.csv.gz"
        txt = out.to_csv(index_label="time_utc", float_format="%.6f")
        with open(f"{OUT_DIR}/{fn}", "wb") as fh:
            with gzip.GzipFile(filename="", mode="wb", fileobj=fh, mtime=0) as gz:
                gz.write(txt.encode())
        pred_files.append(fn)
print(len(pred_files), "prediction files written | rows per file:", {s: int(tab[s]["scoreable"].sum()) for s in ("val", "test")})

14 prediction files written | rows per file: {'val': 12214, 'test': 12180}


## B5. جودة التنبؤ (p.u.)
MAE وRMSE وR² على: clean train، clean val (كله)، clean val قبل الإزاحة، clean val جوا الإزاحة، clean test.
للشبكات: mean ± std (ddof=1) على الـ 3 seeds. المرجع: predictor ثابت = وسيط الـ target في clean train.

In [8]:
def metrics(y, yh):
    e = y - yh
    sst = float(((y - y.mean()) ** 2).sum())
    return {"MAE": float(np.abs(e).mean()), "RMSE": float(np.sqrt((e ** 2).mean())), "R2": 1 - float((e ** 2).sum()) / sst}

SETS = {
    "clean train": clean["train"],
    "clean val": clean["val"],
    "clean val before offset": clean["val"][clean["val"].index < OFFSET_START],
    "clean val inside offset": clean["val"][clean["val"].index >= OFFSET_START],
    "clean test": clean["test"],
}
def yhat_on(name, seed, df):
    return unscale_y(predict_scaled(name, seed, scale_x(df)).astype(np.float64))

per_run = {}
for name, seed in MODELS:
    for sname, df in SETS.items():
        per_run[(name, seed, sname)] = metrics(df[TARGET].to_numpy(), yhat_on(name, seed, df))
const = float(clean["train"][TARGET].median())
for sname, df in SETS.items():
    per_run[("constant", 0, sname)] = metrics(df[TARGET].to_numpy(), np.full(len(df), const))

rows = []
for model in ("constant", "gbm", "mlp", "ae"):
    for sname, df in SETS.items():
        vals = [per_run[k] for k in per_run if k[0] == model and k[2] == sname]
        row = {"model": model, "set": sname, "n_rows": len(df), "n_runs": len(vals)}
        for met in ("MAE", "RMSE", "R2"):
            v = np.array([x[met] for x in vals])
            row[f"{met}_mean"] = v.mean(); row[f"{met}_std"] = v.std(ddof=1) if len(v) > 1 else np.nan
        rows.append(row)
quality = pd.DataFrame(rows)
quality.to_csv(f"{OUT_DIR}/prediction_quality.csv", index=False, float_format="%.6f")
show = quality.copy()
for met in ("MAE", "RMSE", "R2"):
    show[met] = [f"{m:.4f}" + (f" ± {s:.4f}" if np.isfinite(s) else "") for m, s in zip(show[f"{met}_mean"], show[f"{met}_std"])]
print(show[["model", "set", "n_rows", "MAE", "RMSE", "R2"]].to_string(index=False))
assert np.isfinite(quality[[c for c in quality.columns if c.endswith("_mean")]].to_numpy()).all()

   model                     set  n_rows             MAE            RMSE              R2
constant             clean train   83135          0.2153          0.2447         -0.0015
constant               clean val   12146          0.2087          0.2369         -0.0005
constant clean val before offset    7487          0.2202          0.2477         -0.0024
constant clean val inside offset    4659          0.1904          0.2184         -0.0006
constant              clean test    8857          0.2177          0.2471         -0.0216
     gbm             clean train   83135          0.0161          0.0268          0.9880
     gbm               clean val   12146          0.0198          0.0287          0.9853
     gbm clean val before offset    7487          0.0244          0.0342          0.9809
     gbm clean val inside offset    4659          0.0124          0.0166          0.9942
     gbm              clean test    8857          0.0200          0.0259          0.9888
     mlp             

In [9]:
# الفرق بين clean val جوا الإزاحة وقبلها (لكل seed ثم mean ± std)
drows = []
for model in ("constant", "gbm", "mlp", "ae"):
    seeds = sorted({k[1] for k in per_run if k[0] == model})
    row = {"model": model}
    for met in ("MAE", "RMSE", "R2"):
        d = np.array([per_run[(model, s, "clean val inside offset")][met] - per_run[(model, s, "clean val before offset")][met] for s in seeds])
        row[f"d{met} (inside - before)"] = f"{d.mean():+.4f}" + (f" ± {d.std(ddof=1):.4f}" if len(d) > 1 else "")
    drows.append(row)
delta_tbl = pd.DataFrame(drows); print(delta_tbl.to_string(index=False))

   model dMAE (inside - before) dRMSE (inside - before) dR2 (inside - before)
constant                -0.0298                 -0.0293               +0.0018
     gbm                -0.0120                 -0.0176               +0.0133
     mlp       -0.0132 ± 0.0005        -0.0172 ± 0.0005      +0.0131 ± 0.0007
      ae       -0.0110 ± 0.0004        -0.0149 ± 0.0009      +0.0109 ± 0.0007


## B6. المخرجات

In [10]:
gbm.booster_.save_model(f"{OUT_DIR}/models/gbm_best.txt")
for (kind, seed), m in nets.items():
    torch.save(m.state_dict(), f"{OUT_DIR}/models/{kind}_seed{seed}.pt")

training_log = {
    "versions": VERSIONS,
    "gbm": {"runs_by_num_leaves": {str(k): v for k, v in gbm_runs.items()}, "selected_num_leaves": best_nl,
            "feature_importance_gain_pct": imp.to_dict()},
    "smoke_test": SMOKE, "net_runs": runs_log,
    "b1_irradiance_distribution": b1.to_dict(orient="records"),
}
json.dump(training_log, open(f"{OUT_DIR}/training_log.json", "w"), indent=1, default=float)

config_b = {
    "seed_base": SEED, "net_seeds": list(NET_SEEDS), "versions": VERSIONS, "threads": 1, "deterministic_torch": True,
    "gbm_library": "lightgbm", "gbm_fallback_used": False,
    "environment_notes": "lightgbm and torch were not preinstalled and were installed with pip; torch is the CUDA build running on CPU (cuda available: False); LightGBM 4.7.0 emits a deprecation warning for the eval_set argument (no effect on results).",
    "gbm_params": {"objective": "regression", "learning_rate": 0.05, "num_leaves_grid": [15, 31, 63], "min_data_in_leaf": 100,
                   "feature_fraction": 1.0, "n_estimators": 2000, "early_stopping_rounds": 50, "random_state": SEED,
                   "deterministic": True, "force_row_wise": True, "n_jobs": 1},
    "net_architecture": {"encoder": "8-32-16-4 (ReLU, dropout 0.15 after first)", "decoder": "4-16-32-8", "head": "4-8-1"},
    "net_training": {"optimizer": "Adam", "lr": 1e-3, "batch_size": 512, "max_epochs": 50, "patience": 5,
                     "early_stopping_metric": "val pred MSE (clean val, scaled)",
                     "lambdas": {"ae": {"recon": 1, "pred": 5}, "mlp": {"recon": 0, "pred": 5}}},
    "features": FEATURES, "target": TARGET, "scaler": "scaler_clean.json: (x - median) / scale",
    "train_rows": "clean==1 only", "early_stopping_rows": "clean val",
    "r2_std": "ddof=1 over 3 seeds", "constant_predictor": {"value_pu": const, "source": "median of target in clean train"},
    "input_sha256": {f"phase_a/{n}": sha256_file(f"{A_DIR}/{n}") for n in sorted(os.listdir(A_DIR))},
}
json.dump(config_b, open(f"{OUT_DIR}/config_b.json", "w"), indent=1)

# تحقق نهائي من الملفات
assert len(pred_files) == 14 and all(os.path.exists(f"{OUT_DIR}/{f}") for f in pred_files)
for f in pred_files:
    p = pd.read_csv(f"{OUT_DIR}/{f}", index_col="time_utc", parse_dates=True)
    s = f.rsplit("_", 1)[1].split(".")[0]
    assert len(p) == int(tab[s]["scoreable"].sum()) and list(p.columns) == ["y_pu", "yhat_pu", "clean", "in_event_margin"]
    assert p.index.equals(tab[s][tab[s]["scoreable"] == 1].index)
chk = lgb.Booster(model_file=f"{OUT_DIR}/models/gbm_best.txt")
assert np.allclose(chk.predict(Xva), gbm.predict(Xva))
print("B6 asserts passed")
for d in ("preds", "models"):
    print(d, len(os.listdir(f"{OUT_DIR}/{d}")), "files")
for n in ("training_log.json", "prediction_quality.csv", "config_b.json"):
    print(n, f"{os.path.getsize(f'{OUT_DIR}/{n}'):,}", "bytes")

B6 asserts passed
preds 14 files
models 7 files
training_log.json 13,859 bytes
prediction_quality.csv 1,492 bytes
config_b.json 2,277 bytes
